#### You are the Lead Data Analyst at a growing e-commerce platform in Pakistan. 
The management wants to launch a Mega Summer Sale, but they have a massive problem: they don't know which products actually drive value.The raw sales database is a "black box" full of missing entries (logistics errors) and unscaled numbers that make a 10,000 PKR phone look more important than a 500 PKR soap, even if the soap sells 100x more.

### The Mission:

Transform 6 months of "noisy" raw data into a High-Fidelity ML-Ready Handoff. You aren't just "cleaning data"; you are building the mathematical foundation that determines how millions of PKR in marketing budget will be spent.

#### Expected End Results (The Handoff):
1. The Imputed Sales Matrix: A 100% complete dataset where missing values are recovered using cross-month product averages (Smart Recovery).

2. The "Efficiency" Index: A normalized ranking that identifies products over-performing their specific category average (Spotting Trends).

3. The Model Input ($X_{final}$): A zero-mean, unit-variance matrix that is mathematically "healthy" (Condition Number < $10^{10}$) so the Demand Forecasting model doesn't crash during training.Strategic 

### Impact:

#### 1. Marketing: 
You will provide a list of "Category Kings"—products that are in the top 20% of their specific niche, not just the overall platform. These get the homepage banners.

#### 2. Operations: 
You will flag "High-Risk Products"—those with missing data in $>2$ months—to the logistics team for an audit.

#### 3. Finance: 
You will provide the Gram Matrix, allowing the Finance team to see the "co-movement" of sales across months for budget hedging.

##### Problem Statement:
You are a data analyst at a Pakistani e-commerce company. You have received raw sales data for 500 products across 6 months. The data has missing values, outliers, and needs full analysis before it goes to the ML team for demand forecasting.

In [355]:
import numpy as np

np.random.seed(42)

n_products = 500
n_months   = 6

# Raw sales data — some values missing (nan)
sales_raw = np.random.randint(
        100, 10000,
        size=(n_products, n_months)
        ).astype(np.float64)

# Inject 5% missing values randomly
mask = np.random.random(
       size=(n_products, n_months)) < 0.05
sales_raw[mask] = np.nan

# Product categories
categories = np.random.choice(
    ['Electronics', 'Clothing',
     'Food', 'Books', 'Sports'],
    size=n_products)

# Month names
months = ['Jan', 'Feb', 'Mar',
          'Apr', 'May', 'Jun']

In [356]:
most_missing_month = np.argmax( np.isnan(sales_raw).sum(axis=0) )
missing_peroduct_wise = np.isnan(sales_raw).sum(axis=1)

print(f'''
Sales: {sales_raw.shape}, {sales_raw.dtype}, {sales_raw.nbytes} Bytes
Missing values: {( a:= np.isnan(sales_raw).sum() )}, {np.divide(a, sales_raw.size)*100:.2f}%

Most missing value month: {months[most_missing_month]}

Product missing more than 2 months: {np.sum(missing_peroduct_wise > 2)}

{np.shares_memory(sales_raw, mask)}, {np.shares_memory(sales, sales[mask])}
''')



Sales: (500, 6), float64, 24000 Bytes
Missing values: 142, 4.73%

Most missing value month: May

Product missing more than 2 months: 0

False, False



In [357]:
sales = sales_raw.copy()
print(np.shares_memory(sales_raw, sales))

row_means_2d = np.nanmean(sales, axis=1).reshape(-1, 1)
sales = np.where(np.isnan(sales), row_means_2d, sales)

print(f"NaNs remaining: {np.isnan(sales).sum()}") # Verify it's 0
print(f'''
Missing Values: {np.isnan(sales).sum()}
Total Sales per product = \n{ np.round( np.sum(sales, axis = 1), 1) }

Averge Monthly Sales per Product = \n{ np.round( np.mean(sales, axis = 1), 2) } 

Total Monthly Sales across all Product = { np.ceil( np.sum(sales, axis = 0)).astype('i8') }
{np.nanmean(sales).dtype}
''')
# nothing gets up casted as it is already float64

False
NaNs remaining: 0

Missing Values: 0
Total Sales per product = 
[31310.  21846.  27948.  35434.  26998.8 20691.  37162.8 39248.4 31107.
 28024.  28293.  40912.  38941.  41529.  20790.  30985.  34006.  33254.
 35781.  32043.  41070.  31551.  28117.  31824.  35498.4 37004.  32750.4
 19817.  26696.  19501.  38960.  22276.  34800.  29806.8 27501.6 31676.
 29661.  30837.6 31022.4 30286.  27847.  37708.  22453.  27882.  40146.
 31332.  25163.  30743.  51355.5 36535.  11758.  20058.  25901.  22885.
 23360.  27373.5 29889.  31923.6 39114.  38509.  22987.  28546.5 19937.
 42984.  28818.  41719.  29232.  17458.  23482.8 24102.  38084.4 37144.
 36721.  41202.  27788.  33740.4 31220.  33800.  38670.  28225.  38477.
 12846.  36962.  27249.  23856.  27510.  23293.2 36424.  23311.  42806.
 38559.  19174.  33653.  44351.  33637.  24894.  40720.8 47510.  21027.
 13245.  31472.4 24543.6 41056.  30909.6 35171.  32661.6 30958.  26213.
 26520.  32214.  31375.  27164.  33780.  33065.  26192.  35114.  

In [358]:
total_per_product = np.sum(sales, axis = 1)
sorted_indicies = np.argsort(total_per_product)
top10_sales = sorted_indicies[-10:][::-1]
bot10_sales = sorted_indicies[:10]

total_per_month = np.sum(sales, axis = 0)
top_month_index = np.argmax(total_per_month)

best_month_per_product = np.argmax(sales, axis = 1)

print(f'''
Top Sales: \n{ np.round(total_per_product[top10_sales], 2) }

Bottom Sales: \n{ np.round( total_per_product[bot10_sales], 2) }

Highest Sales Month: {months[top_month_index]}


{best_month_per_product.shape}
''')


Top Sales: 
[51355.5 50661.6 48153.6 47983.2 47510.  46929.  46485.  45937.  45498.
 45277.2]

Bottom Sales: 
[ 9529.2 10722.  11758.  12846.  13216.  13245.  15218.4 15396.  15538.
 15690. ]

Highest Sales Month: Jun


(500,)



In [359]:
sale = []
category_benchmarks = np.zeros(500)
total_product_wise = np.sum(sales, axis=1)

category = ['Electronics', 'Clothing', 'Food', 'Books', 'Sports']
for cat in category:
    mask = categories ==cat
    category_mask = categories[mask]
    mean_category = total_product_wise[mask].mean().round(2)
    cat_sales = total_product_wise[mask]
    
    threshold = np.percentile(cat_sales, 80)
    category_benchmarks[mask] = mean_category
    print(f'{cat.title()}: {mean_category} \nthreshold: {threshold.round(1)}\n')
    sale.append(mean_category)

avg_sale = np.array(sale)
total_sale =  np.array(total)
cat_index = np.argmax(avg_sale)
print(f'\nHighest Sold Ttem(avg): {category[cat_index]}')

total_product_wise = np.sum(sales, axis=1)
normalize_version = total_product_wise/category_benchmarks
print(f'''
Above Category Avg (> 1): {(normalize_version > 1).sum()} 
Below Category Avg (< 1): {(normalize_version < 1).sum()} 
''')

Electronics: 29874.75 
threshold: 37004.0

Clothing: 30511.74 
threshold: 36665.0

Food: 29483.88 
threshold: 36049.0

Books: 31240.29 
threshold: 37419.6

Sports: 30765.8 
threshold: 36876.2


Highest Sold Ttem(avg): Books

Above Category Avg (> 1): 239 
Below Category Avg (< 1): 261 



In [360]:
if np.isnan(sales).sum() == 0:
    month_means = sales.mean(axis =0)
    month_stds = sales.std(axis=0)
    x_norm = (sales-month_means)/month_stds

    bias = np.ones((n_product, 1))
    x_final = np.hstack([bias, x_norm])
    print(x_final.shape)

    gram_matrix = np.dot(x_final.T, x_final)
    print(gram_matrix.shape)
    cond = np.linalg.cond(gram_matrix)
    print(f'Condition number: {cond:.2f}') # < 1e10 → well-conditioned ✅


print(f'''
Original Dataset: {sales_raw.nbytes/1e03:.1f} KB
Filled Dataset = {sales.nbytes/1e03:.1f} KB
x_final Dataset: {x_final.nbytes/1e03:.1f} KB
Gram Matrix: {gram_matrix.nbytes/1e03:.1f} KB

''')

total = sales_raw.nbytes + sales.nbytes + x_final.nbytes + gram_matrix.nbytes

print(f'Total RAM: {total/1e03:.2f} KB') 

(500, 7)
(7, 7)
Condition number: 1.39

Original Dataset: 24.0 KB
Filled Dataset = 24.0 KB
x_final Dataset: 28.0 KB
Gram Matrix: 0.4 KB


Total RAM: 76.39 KB
